PHASE 1: Downsampling, Project Generation, & Extraction
Run this block to crunch all the arrival videos to 1080p, build the directory structures, and extract the frames. Once this finishes, you will take the frames to Label Studio.

In [1]:
# ==============================================================================
# RESUME CELL 1: DOWNSAMPLE REMAINDER, CREATE PROJECTS, EXTRACT FRAMES
# ==============================================================================
import os
import glob
import deeplabcut

WORKING_DIR = '/mnt/Data/Projects/cloud_deployment/AprilProjects/'
VIDEO_DIR = '/mnt/Data/Projects/cloud_deployment/videos/SER/'

# Find the arrival videos
raw_videos = glob.glob(os.path.join(VIDEO_DIR, '*arrival*.MP4'))
# Exclude any that are already 1080p to prevent redundant processing
raw_videos = [v for v in raw_videos if '1080p' not in v]

print(f"🔎 Found {len(raw_videos)} raw arrival videos.")

downsampled_videos = []

# --- 1. SMART DOWNSAMPLE LOOP (Picks up where you canceled) ---
for vid in raw_videos:
    out_path = vid.replace('.MP4', '_1080p.MP4')
    downsampled_videos.append(out_path)
    
    if not os.path.exists(out_path):
        print(f"🎬 Crunching to 1080p: {os.path.basename(vid)}")
        os.system(f'ffmpeg -hide_banner -loglevel warning -y -i "{vid}" -vf scale=1920:1080 -c:v libx264 -crf 18 -preset fast -c:a copy "{out_path}"')
    else:
        print(f"⏭️ 1080p already exists, skipping conversion: {os.path.basename(out_path)}")

# --- 2. PROJECT ROUTING & CREATION ---
# We map the video suffix to the correct DLC project name
project_map = {
    'tp': 'EagleRay_Top_Base', # Existing
    'w1': 'EagleRay_UW1',      # New
    'w2': 'EagleRay_UW2',      # New
    'w3': 'EagleRay_UW3'       # New
}

configs = {}

for vid in downsampled_videos:
    vid_name = os.path.basename(vid)
    view_key = next((key for key in project_map.keys() if key in vid_name), None)
    
    if not view_key:
        continue
        
    proj_name = project_map[view_key]
    
    if view_key == 'tp':
        # Load Existing Top Project
        cfg_path = glob.glob(os.path.join(WORKING_DIR, f'{proj_name}-Dev-*', 'config.yaml'))[0]
        print(f"\n📂 Loading existing Top project...")
        # DLC will safely ignore this if the video was already added before you canceled
        deeplabcut.add_new_videos(cfg_path, [vid], copy_videos=False)
        configs[view_key] = cfg_path
    else:
        # Create New Underwater Projects
        print(f"\n🚀 Checking/Creating project for {view_key.upper()}...")
        existing = glob.glob(os.path.join(WORKING_DIR, f'{proj_name}-Dev-*', 'config.yaml'))
        if existing:
            print("⏭️ Project already exists, loading config...")
            cfg_path = existing[0]
        else:
            cfg_path = deeplabcut.create_new_project(proj_name, 'Dev', [vid], working_directory=WORKING_DIR, copy_videos=False)
        configs[view_key] = cfg_path

    # --- 3. FRAME EXTRACTION ---
    print(f"📸 Extracting frames for {view_key.upper()}...")
    deeplabcut.extract_frames(configs[view_key], mode='automatic', algo='kmeans', userfeedback=False)

print("\n✅ Extraction complete for all 4 views. Proceed to Label Studio.")

2026-04-10 00:03:34.260218: I tensorflow/core/util/port.cc:110] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-04-10 00:03:34.355150: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-04-10 00:03:35.030047: W tensorflow/compiler/tf2tensorrt/utils/py_utils.cc:38] TF-TRT Warning: Could not find TensorRT


Loading DLC 2.3.11...
DLC loaded in light mode; you cannot use any GUI (labeling, relabeling and standalone GUI)


/home/marno/miniconda3/envs/dlc/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


🔎 Found 4 raw arrival videos.
⏭️ 1080p already exists, skipping conversion: Fem1_OFt1arrivalw2_20260401_GX010284-002_1080p.MP4
⏭️ 1080p already exists, skipping conversion: Fem1_OFt1arrivalw1_20260401_GX010284-001_1080p.MP4
⏭️ 1080p already exists, skipping conversion: Fem1_OFt1arrivaltp_20260401_GX010184_1080p.MP4
⏭️ 1080p already exists, skipping conversion: Fem1_OFt1arrivalw3_20260401_GX010052-003_1080p.MP4

🚀 Checking/Creating project for W2...
⏭️ Project already exists, loading config...
📸 Extracting frames for W2...
Config file read successfully.
Extracting frames based on kmeans ...
Kmeans-quantization based extracting of frames from 105.0  seconds to 896.9  seconds.
Extracting and downsampling... 23734  frames from the video.


919it [00:08, 114.16it/s]


KeyboardInterrupt: 

PHASE 2: The Multi-Camera Parser & Trainer
After you finish labeling the frames for all the views, export the CSVs from Label Studio. Update the LABEL_STUDIO_CSVS dictionary in the code below with the paths to your new CSVs.

This block will parse them all, stitch the old and new data together for the Top view, patch the VRAM parameters, and train/analyze all the models sequentially.

In [2]:
# ==============================================================================
# CELL 2: MULTI-VIEW PARSING & BATCH TRAINING (PATCHED FOR 4-POINT)
# ==============================================================================
import os
import glob
import pandas as pd
import numpy as np
import json
import yaml
import ruamel.yaml
import tensorflow as tf
import deeplabcut

# --- 1. USER INPUTS ---
# Update these paths with your exported CSVs from Label Studio
LABEL_STUDIO_CSVS = {
    'tp': '/mnt/Data/Projects/cloud_deployment/AprilProjects/EagleRay_Top_Base-Dev-2026-04-08/labeled-data/Fem1_OFt1arrivaltp_20260401_GX010184_1080p/project-17-at-2026-04-09-22-22-c9537410.csv',
    'w1': '/mnt/Data/Projects/cloud_deployment/AprilProjects/EagleRay_UW1-Dev-2026-04-09/labeled-data/Fem1_OFt1arrivalw1_20260401_GX010284-001_1080p/project-18w1-at-2026-04-10-00-40-b2c274e6.csv',
    'w2': '/mnt/Data/Projects/cloud_deployment/AprilProjects/EagleRay_UW2-Dev-2026-04-09/labeled-data/Fem1_OFt1arrivalw2_20260401_GX010284-002_1080p/project-19w2-at-2026-04-10-00-41-ef33a523.csv',
    'w3': '/mnt/Data/Projects/cloud_deployment/AprilProjects/EagleRay_UW3-Dev-2026-04-09/labeled-data/Fem1_OFt1arrivalw3_20260401_GX010052-003_1080p/project-20w3-at-2026-04-10-00-43-adc6cb3f.csv'
}

# The Amputated 4-Point Skeleton
BODYPARTS = ['Snout', 'LeftWingtip', 'RightWingtip', 'TailBase'] 
SKELETON = [
    ['Snout', 'LeftWingtip'], 
    ['Snout', 'RightWingtip'], 
    ['LeftWingtip', 'TailBase'], 
    ['RightWingtip', 'TailBase']
]

WORKING_DIR = '/mnt/Data/Projects/cloud_deployment/AprilProjects/'

os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2' 
os.environ['TF_GPU_ALLOCATOR'] = 'cuda_malloc_async'
gpus = tf.config.list_physical_devices('GPU')
if gpus: tf.config.experimental.set_memory_growth(gpus[0], True)

project_map = {'tp': 'EagleRay_Top_Base', 'w1': 'EagleRay_UW1', 'w2': 'EagleRay_UW2', 'w3': 'EagleRay_UW3'}
ryaml = ruamel.yaml.YAML()
ryaml.preserve_quotes = True

for view_key, csv_path in LABEL_STUDIO_CSVS.items():
    if not os.path.exists(csv_path):
        print(f"⚠️ Skipping {view_key}: CSV not found at {csv_path}")
        continue
        
    proj_name = project_map[view_key]
    cfg_path = glob.glob(os.path.join(WORKING_DIR, f'{proj_name}-Dev-*', 'config.yaml'))[0]
    proj_dir = os.path.dirname(cfg_path)
    
    # --- 2. ENFORCE 4-POINT SKELETON IN CONFIG ---
    print(f"\n🦴 Injecting custom 4-Point Skeleton into {view_key.upper()} config...")
    with open(cfg_path, 'r') as f:
        cfg = ryaml.load(f)
    
    cfg['bodyparts'] = BODYPARTS
    cfg['skeleton'] = SKELETON
    scorer = cfg.get('scorer', 'Dev')
    
    with open(cfg_path, 'w') as f:
        ryaml.dump(cfg, f)
    
    video_folders = [d for d in os.listdir(os.path.join(proj_dir, 'labeled-data')) if 'arrival' in d]
    if not video_folders: continue
    video_folder = video_folders[0]
    
    # --- 3. GOLDEN PARSER (4-Point Only) ---
    print(f"⚙️ Parsing Label Studio CSV for {view_key.upper()}...")
    df_ls = pd.read_csv(csv_path)
    json_col = next((col for col in ['label', 'annotation', 'keypointlabels'] if col in df_ls.columns), None)
    
    actual_img_dir = os.path.join(proj_dir, 'labeled-data', video_folder)
    physical_files = os.listdir(actual_img_dir)
    file_map = {int(''.join(filter(str.isdigit, f))): f for f in physical_files if f.startswith('img') and f.endswith('.png')}

    data_dict = []
    for idx, row in df_ls.iterrows():
        labels_str = row.get(json_col, '')
        img_path_raw = str(row.get('image', ''))
        
        fname_raw = os.path.basename(img_path_raw).split('-')[-1] if '-' in img_path_raw else os.path.basename(img_path_raw)
        frame_num = int(''.join(filter(str.isdigit, fname_raw)))
        fname = file_map.get(frame_num, fname_raw)
            
        rel_path = f"labeled-data/{video_folder}/{fname}"
        frame_data = {'image': rel_path}
        for bp in BODYPARTS: frame_data[f'{bp}_x'], frame_data[f'{bp}_y'] = np.nan, np.nan
            
        if pd.notna(labels_str) and isinstance(labels_str, str):
            try:
                labels = json.loads(labels_str)
                if isinstance(labels, list) and len(labels) > 0 and 'result' in labels[0]: labels = labels[0]['result']
                for label in labels:
                    val = label.get('value', label) 
                    if 'keypointlabels' in val and len(val['keypointlabels']) > 0:
                        bp = val['keypointlabels'][0]
                        if bp in BODYPARTS:
                            frame_data[f'{bp}_x'] = (val['x'] * 1920) / 100.0
                            frame_data[f'{bp}_y'] = (val['y'] * 1080) / 100.0
            except Exception: pass
        data_dict.append(frame_data)

    df_video = pd.DataFrame(data_dict).set_index('image')
    multi_cols = pd.MultiIndex.from_product([[scorer], BODYPARTS, ['x', 'y']], names=['scorer', 'bodyparts', 'coords'])
    df_final = pd.concat([df_video[f'{bp}_{coord}'] for bp in BODYPARTS for coord in ['x', 'y']], axis=1)
    df_final.columns = multi_cols

    df_final.to_hdf(os.path.join(actual_img_dir, f'CollectedData_{scorer}.h5'), key='df_with_missing', mode='w')
    df_final.to_csv(os.path.join(actual_img_dir, f'CollectedData_{scorer}.csv'))
    
    # --- 4. DATASET BUILD & CONFIG ROUTING ---
    print(f"🧹 Creating Training Dataset for {view_key.upper()}...")
    deeplabcut.create_training_dataset(cfg_path, net_type='resnet_50')
    
    pose_configs = glob.glob(os.path.join(proj_dir, 'dlc-models', '*', '*', '*', 'pose_cfg.yaml'), recursive=True)
    
    for p in pose_configs:
        with open(p, 'r') as f:
            pose_cfg = yaml.safe_load(f)
            
        pose_cfg['dataset_type'] = 'default' 
        pose_cfg['batch_size'] = 1 
        pose_cfg['global_scale'] = 0.8
            
        with open(p, 'w') as f:
            yaml.dump(pose_cfg, f, default_flow_style=False)

    # --- 5. BATCH TRAIN & INFERENCE ---
    print(f"🔥 Training {view_key.upper()} Model...")
    deeplabcut.train_network(cfg_path, maxiters=50000, displayiters=1000, saveiters=10000, allow_growth=True) 
    
    print(f"🧠 Analyzing {view_key.upper()} Video...")
    # Find the corresponding arrival video
    arrival_vids = glob.glob(os.path.join('/mnt/Data/Projects/cloud_deployment/videos/SER/', f'*arrival*{view_key}*1080p.MP4'))
    
    # Fallback to parts if the main video was split
    if not arrival_vids:
         arrival_vids = glob.glob(os.path.join('/mnt/Data/Projects/cloud_deployment/videos/SER/', f'Part1_arrival*{view_key}*.MP4')) + \
                        glob.glob(os.path.join('/mnt/Data/Projects/cloud_deployment/videos/SER/', f'Part2_arrival*{view_key}*.MP4'))
         
    if arrival_vids:
        deeplabcut.analyze_videos(cfg_path, arrival_vids, save_as_csv=True, batchsize=1)
        deeplabcut.create_labeled_video(cfg_path, arrival_vids, draw_skeleton=True)
    else:
        print(f"⚠️ Could not find 1080p arrival video for {view_key} to analyze.")
        
    tf.keras.backend.clear_session()

print("\n✅ All models trained and videos analyzed.")

2026-04-10 00:03:55.699730: I tensorflow/compiler/xla/stream_executor/cuda/cuda_gpu_executor.cc:996] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysfs-bus-pci#L344-L355
2026-04-10 00:03:55.703908: I tensorflow/compiler/xla/stream_executor/cuda/cuda_gpu_executor.cc:996] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysfs-bus-pci#L344-L355
2026-04-10 00:03:55.706993: I tensorflow/compiler/xla/stream_executor/cuda/cuda_gpu_executor.cc:996] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysf


🦴 Injecting custom 4-Point Skeleton into TP config...
⚙️ Parsing Label Studio CSV for TP...
🧹 Creating Training Dataset for TP...
The training dataset is successfully created. Use the function 'train_network' to start training. Happy training!
🔥 Training TP Model...
Selecting single-animal trainer
Batch Size is 1


2026-04-10 00:03:57.348858: I tensorflow/compiler/xla/stream_executor/cuda/cuda_gpu_executor.cc:996] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysfs-bus-pci#L344-L355
2026-04-10 00:03:57.352354: I tensorflow/compiler/xla/stream_executor/cuda/cuda_gpu_executor.cc:996] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysfs-bus-pci#L344-L355
2026-04-10 00:03:57.355230: I tensorflow/compiler/xla/stream_executor/cuda/cuda_gpu_executor.cc:996] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysf

Loading ImageNet-pretrained resnet_50


2026-04-10 00:03:58.020603: I tensorflow/compiler/xla/stream_executor/cuda/cuda_gpu_executor.cc:996] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysfs-bus-pci#L344-L355
2026-04-10 00:03:58.022505: I tensorflow/compiler/xla/stream_executor/cuda/cuda_gpu_executor.cc:996] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysfs-bus-pci#L344-L355
2026-04-10 00:03:58.023991: I tensorflow/compiler/xla/stream_executor/cuda/cuda_gpu_executor.cc:996] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysf

...

Saving results in /mnt/Data/Projects/cloud_deployment/videos/SER...
Saving csv poses!
The videos are analyzed. Now your research can truly start! 
 You can create labeled videos with 'create_labeled_video'
If the tracking is not satisfactory for some videos, consider expanding the training set. You can use the function 'extract_outlier_frames' to extract a few representative outlier frames.


/home/marno/miniconda3/envs/dlc/lib/python3.10/site-packages/deeplabcut/utils/auxiliaryfunctions.py:403: FutureWarning: Starting with pandas version 3.0 all arguments of to_hdf except for the argument 'path_or_buf' will be keyword-only.
  DataMachine.to_hdf(dataname, "df_with_missing", format="table", mode="w")


Starting to process video: /mnt/Data/Projects/cloud_deployment/videos/SER/Fem1_OFt1arrivalw3_20260401_GX010052-003_1080p.MP4
Loading /mnt/Data/Projects/cloud_deployment/videos/SER/Fem1_OFt1arrivalw3_20260401_GX010052-003_1080p.MP4 and data.
Duration of video [s]: 901.23, recorded with 29.97 fps!
Overall # of frames: 27010 with cropped frame dimensions: 1920 1080
Generating frames and creating video.


100%|██████████| 27010/27010 [05:25<00:00, 82.88it/s]



✅ All models trained and videos analyzed.


PHASE 3: Raw Kinematics Plotter
Once a video is analyzed, you can pass the resulting CSV through this block. It removes the ML Imputation logic, drops any points with < 0.6 confidence or that jump more than 50 pixels, and plots the raw physical data.

In [ ]:
# ==============================================================================
# CELL 3: RAW KINEMATICS (NO IMPUTATION)
# ==============================================================================
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Point this to whichever CSV you want to check (tp, w1, w2, w3)
ANALYSIS_CSV = '/path/to/your/analyzed/videoDLC_resnet50_EagleRay_Top_BaseApr8shuffle1_50000.csv'
FPS = 30           

print("⚙️ Loading raw kinematics data...")
df = pd.read_csv(ANALYSIS_CSV, header=[0, 1, 2], index_col=0)
scorer = df.columns.get_level_values(0)[0]
bodyparts = df.columns.get_level_values(1).unique()
time_sec = np.arange(len(df)) / FPS

# --- 1. THE PURGE (Filter out garbage) ---
print("🧹 Purging low confidence and teleporting frames...")
for bp in bodyparts:
    # 1. Confidence filter
    bad_conf = df[scorer, bp, 'likelihood'] < 0.6
    df.loc[bad_conf, (scorer, bp, 'x')] = np.nan
    df.loc[bad_conf, (scorer, bp, 'y')] = np.nan
    
    # 2. Velocity filter (Teleportation prevention)
    dist = np.sqrt(df[scorer, bp, 'x'].diff()**2 + df[scorer, bp, 'y'].diff()**2)
    jumps = dist > 50
    df.loc[jumps, (scorer, bp, 'x')] = np.nan
    df.loc[jumps, (scorer, bp, 'y')] = np.nan

# --- 2. CALCULATE METRICS (With NaNs ignored) ---
print("📐 Calculating Biomechanical Metrics...")
com_x = df.xs('x', level=2, axis=1).mean(axis=1, skipna=True)
com_y = df.xs('y', level=2, axis=1).mean(axis=1, skipna=True)

dx = com_x.diff()
dy = com_y.diff()
velocity_px_s = (np.sqrt(dx**2 + dy**2)) * FPS

# Heading / Orientation Angle 
snout_x, snout_y = df[scorer, 'Snout', 'x'], df[scorer, 'Snout', 'y']
tail_x, tail_y = df[scorer, 'TailBase', 'x'], df[scorer, 'TailBase', 'y']
# We only calculate heading where both points exist
valid_heading = snout_x.notna() & tail_x.notna()
heading_deg = pd.Series(np.nan, index=df.index)
heading_deg[valid_heading] = np.degrees(np.unwrap(np.arctan2(snout_y[valid_heading] - tail_y[valid_heading], snout_x[valid_heading] - tail_x[valid_heading])))

# Wing Flap Proxy 
lw_x, lw_y = df[scorer, 'LeftWingtip', 'x'], df[scorer, 'LeftWingtip', 'y']
rw_x, rw_y = df[scorer, 'RightWingtip', 'x'], df[scorer, 'RightWingtip', 'y']
wing_span_px = np.sqrt((lw_x - rw_x)**2 + (lw_y - rw_y)**2)

# --- 3. PLOTTING PUBLICATION FIGURE ---
print("📊 Generating Plot...")
plt.style.use('seaborn-v0_8-darkgrid')
fig, axes = plt.subplots(2, 2, figsize=(16, 10))
fig.suptitle("Raw Kinematics Report (No Imputation)", fontsize=18, fontweight='bold', y=0.98)

# Panel 1: Tank Trajectory
ax1 = axes[0, 0]
# Drop NaNs just for the KDE plot
valid_com = pd.DataFrame({'x': com_x, 'y': com_y}).dropna()
if not valid_com.empty:
    sns.kdeplot(x=valid_com['x'], y=valid_com['y'], cmap="mako", fill=True, thresh=0.05, alpha=0.5, ax=ax1)
ax1.plot(com_x, com_y, color='red', linewidth=1.2, alpha=0.7)
ax1.set_title("1. Center of Mass Path", fontsize=12, fontweight='bold')
ax1.invert_yaxis()

# Panel 2: Velocity
ax2 = axes[0, 1]
ax2.plot(time_sec, velocity_px_s, color='darkorange', linewidth=1.5)
ax2.set_title("2. Absolute Velocity (px/s)", fontsize=12, fontweight='bold')

# Panel 3: Heading Angle
ax3 = axes[1, 0]
ax3.plot(time_sec, heading_deg, color='purple', linewidth=1.5)
ax3.set_title("3. Unwrapped Heading / Orientation", fontsize=12, fontweight='bold')

# Panel 4: Wing Flap Amplitude
ax4 = axes[1, 1]
ax4.plot(time_sec, wing_span_px, color='teal', linewidth=1.5)
ax4.set_title("4. Apparent Wing Span (px)", fontsize=12, fontweight='bold')

plt.tight_layout(rect=[0, 0, 1, 0.96])
plt.show()

Frame re-extraction for water views preventing training poisoning by frames looking at the ceiling. 

In [4]:
# ==============================================================================
# TARGETED RE-EXTRACTION: UW VIEWS (UNIFORM ALGO, START AT 1:45)
# ==============================================================================
import os
import glob
import cv2
import shutil
import ruamel.yaml
import deeplabcut

WORKING_DIR = '/mnt/Data/Projects/cloud_deployment/AprilProjects/'
START_TIME_SEC = 105  # 1 minute 45 seconds
FRAMES_TO_PICK = 120

project_map = {'w1': 'EagleRay_UW1', 'w2': 'EagleRay_UW2', 'w3': 'EagleRay_UW3'}
ryaml = ruamel.yaml.YAML()
ryaml.preserve_quotes = True

for view_key, proj_name in project_map.items():
    print(f"\n🌊 Processing {view_key.upper()}...")
    
    cfg_paths = glob.glob(os.path.join(WORKING_DIR, f'{proj_name}-Dev-*', 'config.yaml'))
    if not cfg_paths:
        print(f"⚠️ Could not find project for {proj_name}. Skipping.")
        continue
    cfg_path = cfg_paths[0]
    proj_dir = os.path.dirname(cfg_path)

    # --- 1. NUKE DIRECTORIES TO FORCE EXTRACTION ---
    labeled_data_dir = os.path.join(proj_dir, 'labeled-data')
    if os.path.exists(labeled_data_dir):
        for video_folder in os.listdir(labeled_data_dir):
            folder_path = os.path.join(labeled_data_dir, video_folder)
            if os.path.isdir(folder_path):
                shutil.rmtree(folder_path) # Deletes the entire folder structure
                print(f"🧹 Nuked existing directory: {video_folder}")

    # --- 2. CALCULATE TIME BOUNDS ---
    with open(cfg_path, 'r') as f:
        cfg = ryaml.load(f)

    video_dict = cfg.get('video_sets', {})
    if not video_dict:
        continue
    video_path = list(video_dict.keys())[0]
    
    cap = cv2.VideoCapture(video_path)
    fps = cap.get(cv2.CAP_PROP_FPS)
    total_frames = cap.get(cv2.CAP_PROP_FRAME_COUNT)
    cap.release()

    duration = total_frames / fps
    start_fraction = START_TIME_SEC / duration

    if start_fraction >= 1.0:
        print(f"⚠️ Video is shorter than 1:45. Setting start to 0.")
        start_fraction = 0.0

    # --- 3. PATCH CONFIG ---
    cfg['numframes2pick'] = FRAMES_TO_PICK
    cfg['start'] = float(start_fraction)
    cfg['stop'] = 1.0 # Ensure it calculates to the very end of the video

    with open(cfg_path, 'w') as f:
        ryaml.dump(cfg, f)
    print(f"🔧 Configured for {FRAMES_TO_PICK} frames from {start_fraction*100:.1f}% to 100%.")

    # --- 4. EXTRACT (UNIFORM) ---
    print(f"📸 Extracting strictly uniform timeline...")
    deeplabcut.extract_frames(cfg_path, mode='automatic', algo='uniform', userfeedback=False)

print("\n✅ Clean re-extraction complete. The new frames are ready for Label Studio.")


🌊 Processing W1...
🧹 Nuked existing directory: Fem1_OFt1arrivalw1_20260401_GX010284-001_1080p
🔧 Configured for 120 frames from 11.7% to 100%.
📸 Extracting strictly uniform timeline...
Config file read successfully.
Extracting frames based on uniform ...
Uniformly extracting of frames from 105.0  seconds to 896.9  seconds.
Frames were successfully extracted, for the videos listed in the config.yaml file.

You can now label the frames using the function 'label_frames' (Note, you should label frames extracted from diverse videos (and many videos; we do not recommend training on single videos!)).

🌊 Processing W2...
🧹 Nuked existing directory: Fem1_OFt1arrivalw2_20260401_GX010284-002_1080p
🔧 Configured for 120 frames from 11.7% to 100%.
📸 Extracting strictly uniform timeline...
Config file read successfully.
Extracting frames based on uniform ...
Uniformly extracting of frames from 105.0  seconds to 896.9  seconds.
Frames were successfully extracted, for the videos listed in the config.ya

In [ ]:
# ==============================================================================
# KINEMATIC CROSS-CORRELATION (AUTOMATIC VIDEO SYNC)
# ==============================================================================
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy import signal

# --- 1. CONFIG ---
# We use the Top view as "Frame 0" (Master Clock), and sync W3 to it.
FILE_TP = 'Fem1_OFt1arrivaltp_20260401_GX010184_1080pDLC_resnet50_EagleRay_Top_BaseApr8shuffle1_50000.csv'
FILE_W3 = 'Fem1_OFt1arrivalw3_20260401_GX010052-003_1080pDLC_resnet50_EagleRay_UW3Apr9shuffle1_50000.csv'

# --- 2. SIGNAL EXTRACTION (WINGSPAN) ---
def get_clean_wingspan(csv_file):
    df = pd.read_csv(csv_file, header=[0, 1, 2], index_col=0)
    scorer = df.columns.get_level_values(0)[0]
    
    lw_x, lw_y = df[scorer, 'LeftWingtip', 'x'], df[scorer, 'LeftWingtip', 'y']
    rw_x, rw_y = df[scorer, 'RightWingtip', 'x'], df[scorer, 'RightWingtip', 'y']
    
    # Only keep data where the network is highly confident
    lw_conf = df[scorer, 'LeftWingtip', 'likelihood']
    rw_conf = df[scorer, 'RightWingtip', 'likelihood']
    mask = (lw_conf > 0.6) & (rw_conf > 0.6)
    
    # Calculate Euclidean distance between wingtips
    span = np.sqrt((lw_x - rw_x)**2 + (lw_y - rw_y)**2)
    span[~mask] = np.nan
    
    # Smooth over small missing gaps (like when the ray turns away)
    # Fill the giant empty gaps (the first 9 minutes) with 0 so the math doesn't break
    span = span.interpolate(method='pchip', limit=15).fillna(0)
    
    # Normalize the wave (center it at 0 for correlation)
    span = span - span.mean()
    return span.values

print("⚙️ Loading Top and W3 kinematic profiles...")
span_tp = get_clean_wingspan(FILE_TP)
span_w3 = get_clean_wingspan(FILE_W3)

# --- 3. CROSS-CORRELATION ---
print("🔬 Running 1D Cross-Correlation to find biological sync point...")

# This slides the W3 wave across the TP wave to find the maximum overlap
correlation = signal.correlate(span_tp, span_w3, mode='full')
lags = signal.correlation_lags(len(span_tp), len(span_w3), mode='full')

# Isolate the mathematical peak
best_lag_index = np.argmax(correlation)
offset_frames = lags[best_lag_index]

print(f"\n✅ SYNC FOUND: Camera W3 is offset by exactly {offset_frames} frames relative to the Top camera.")

# --- 4. PLOTTING THE ALIGNMENT VERIFICATION ---
plt.style.use('seaborn-v0_8-darkgrid')
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 8))
fig.suptitle("Kinematic Syncing via Cross-Correlation", fontsize=16, fontweight='bold')

# PANEL 1: Before alignment (The raw mess)
ax1.plot(span_tp, label='Top Camera (tp)', alpha=0.8, color='blue')
ax1.plot(span_w3, label='Underwater 3 (w3)', alpha=0.8, color='orange')
ax1.set_title("Raw Timestamps (Unsynced)")
ax1.set_ylabel("Normalized Wingspan Variance")
ax1.legend()

# PANEL 2: After alignment (Biologically locked)
ax2.plot(span_tp, label='Top Camera Master Clock (tp)', alpha=0.8, color='blue')
# Create a shifted x-axis for w3 by adding the calculated offset
shifted_x = np.arange(len(span_w3)) + offset_frames
ax2.plot(shifted_x, span_w3, label=f'Underwater 3 (Shifted by {offset_frames} frames)', alpha=0.8, color='green')
ax2.set_title("Biologically Synced")
ax2.set_ylabel("Normalized Wingspan Variance")
ax2.set_xlabel("Frame Index (Relative to Top Camera)")
ax2.legend()

plt.tight_layout()
plt.show()